In [ ]:
# ── Robot connection ──────────────────────────────────────────────────────────
# Normal mode: running notebooks directly on the Reachy host → use 'localhost'.
# Remote mode: set REACHY_IP to the robot's IP address before connecting.
# REACHY_IP = "10.22.129.133"   # physical robot (remote)
R_IP = "10.22.129.133"         # running on Reachy host (default)


In [1]:
import os
import glob
import cv2
from reachy_sdk import ReachySDK

# 1. Configuration
ROBOT_IP = '10.22.129.133' 
CAMERA_EYE = 'right'     

# Define path and instantly expand the tilde (~) symbol
RAW_PATH = '~/dev/reachy-2026-iitg/reachy-tabletop-ai/data/calibration/annotation/cube/'
OUTPUT_FOLDER = os.path.expanduser(RAW_PATH) 

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    print(f"Created folder: '{OUTPUT_FOLDER}'")

# 2. Smart Counter Setup
existing_files = glob.glob(os.path.join(OUTPUT_FOLDER, "image_*.jpg"))

if existing_files:
    existing_numbers = []
    for filepath in existing_files:
        filename = os.path.basename(filepath)  # e.g., "image_000.jpg"
        try:
            number_part = filename.replace("image_", "").replace(".jpg", "")
            existing_numbers.append(int(number_part))
        except ValueError:
            continue
    # Start at the highest found number + 1
    image_counter = max(existing_numbers, default=0) + 1
    print(f"Found existing images. Starting new images at index: {image_counter:03d}")
else:
    image_counter = 0
    print("Folder is empty. Starting fresh at index: 000")

# 3. Connect to Reachy
print(f"Connecting to Reachy at {ROBOT_IP}...")
try:
    robot = ReachySDK(host=ROBOT_IP)
    print("Successfully connected to the robot!")
except Exception as e:
    print(f"Connection failed: {e}")
    exit()

if CAMERA_EYE == 'left':
    camera_feed = robot.left_camera
else:
    camera_feed = robot.right_camera

print("\n--- Image Collection Controls ---")
print("  Press 's' to Save an image")
print("  Press 'q' to Quit the script")
print("---------------------------------\n")

try:
    while True:
        frame = camera_feed.last_frame
        if frame is None:
            continue
            
        display_frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        cv2.imshow("Reachy Eye Feed - Dataset Collection", display_frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('s'):
            # This uses the exact same OUTPUT_FOLDER and image_counter variables
            image_filename = os.path.join(OUTPUT_FOLDER, f"image_{image_counter:03d}.jpg")
            cv2.imwrite(image_filename, display_frame)
            print(f"Saved: {image_filename}")
            image_counter += 1

        elif key == ord('q'):
            print("Stopping collection script.")
            break

except KeyboardInterrupt:
    print("\nScript interrupted by user.")

finally:
    cv2.destroyAllWindows()
    print(f"Finished! Collected images are in '{OUTPUT_FOLDER}'.")


/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Found existing images. Starting new images at index: 001
Connecting to Reachy at 10.22.129.133...
Successfully connected to the robot!

--- Image Collection Controls ---
  Press 's' to Save an image
  Press 'q' to Quit the script
---------------------------------

Stopping collection script.
Finished! Collected images are in '/home/reachy/dev/reachy-2026-iitg/reachy-tabletop-ai/data/calibration/annotation/cube/'.
